# Optimize temporal synthetic data parameters

This notebook searches for a temporal synthetic data-generating process such that:

- the **IID** engine declares equivalence,
- the **cluster-aware** engine declares equivalence, and
- the **temporal** engine does **not** declare equivalence.

The notebook follows the same structure and helper logic used by the other synthetic-data notebooks.


## 1. Environment setup


In [ ]:
from pyTOST.data_gen.notebook_utils import (
    configure_notebook_environment,
    summarize_result,
    history_frame,
    result_payload,
    save_json,
)
from pyTOST.data_gen.optimize_temporal_params import SearchSpace, search, evaluate_params

REPO_ROOT = configure_notebook_environment()
ALPHA = 0.05
MARGIN = 1.0
SEARCH_SEED = 123
VALIDATION_SEED = 2026
OUTPUT_JSON = "best_temporal_params.json"
HAC_LAGS = 8

ENGINE_SPECS = {
    "IID": {"ci": "ci_iid", "eq": "eq_iid", "mu": "mu_hat_iid"},
    "Cluster": {"ci": "ci_cluster", "eq": "eq_cluster", "mu": "mu_hat_cluster"},
    "Temporal": {"ci": "ci_temporal", "eq": "eq_temporal", "mu": "mu_hat_temporal"},
}


## 2. Configure the optimization problem


In [ ]:
space = SearchSpace(
    n_time_min=200,
    n_time_max=900,
    series_min=1,
    series_max=4,
    rho_min=0.93,
    rho_max=0.995,
    process_sd_min=0.1,
    process_sd_max=2.0,
    obs_sd_min=0.01,
    obs_sd_max=0.6,
    delta_true_min=0.9,
    delta_true_max=0.995,
)
N_ITER = 600
VERBOSE_EVERY = 25


## 3. Run the search


In [ ]:
best, history = search(
    seed=SEARCH_SEED,
    n_iter=N_ITER,
    alpha=ALPHA,
    margin=MARGIN,
    hac_lags=HAC_LAGS,
    space=space,
    verbose_every=VERBOSE_EVERY,
)

best_summary = summarize_result(best, ENGINE_SPECS)
best_summary


## 4. Summarize the best candidate


In [ ]:
best_payload = result_payload(
    params=best.params,
    summary_source=best,
    engine_specs=ENGINE_SPECS,
    extra={"validation_seed": VALIDATION_SEED, "hac_lags": HAC_LAGS},
)
best_payload


## 5. Save the best parameters


In [ ]:
save_json(best_payload, OUTPUT_JSON)
OUTPUT_JSON


## 6. Validate by regenerating data and rerunning the target engines


In [ ]:
validation = evaluate_params(
    best.params,
    seed=VALIDATION_SEED,
    alpha=ALPHA,
    margin=MARGIN,
    hac_lags=HAC_LAGS,
)
validation_summary = summarize_result(validation, ENGINE_SPECS)
validation_summary


## 7. Search diagnostics


In [ ]:
history_df = history_frame(
    history,
    ENGINE_SPECS,
    param_keys=["n_time", "series_per_arm", "rho", "process_sd", "obs_sd", "delta"],
)
history_df.sort_values("score").head(10)
